In [1]:
import os
import sys

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED, init_plsa

In [4]:
import artm
from artm import ARTM, Dictionary

import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

from topicnet.cooking_machine.models.topic_model import ARTM_NINE
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.viewers.top_documents_viewer import TopDocumentsViewer
from topicnet.viewers.top_tokens_viewer import TopTokensViewer
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    count_vocab_size,
    init_model,
)
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    modality_weight_rel2abs,
    transform_regularizer,
)


import numpy as np
import pandas as pd
from pandas import DataFrame
from scipy.spatial.distance import cdist

import os
import tempfile
import warnings
from copy import deepcopy
from typing import Dict, List, Optional

In [5]:
from IPython.display import display, display_html

In [6]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [7]:
! ls $DATA_FOLDER_PATH

20NG.csv	 MKB10__internals      RTL_Wiki_person.csv
20NG__internals  postnauka.csv	       RTL_Wiki_person__internals
api.py		 postnauka__internals  ruwiki_good__internals
Brown		 __pycache__	       ruwiki_good.txt
Brown_BOW.csv	 Reuters	       WikiRef-220
Brown_NOOW.csv	 Reuters_BOW.csv       wiki_ref220_bow.csv
__init__.py	 Reuters_NOOW.csv      wiki_ref220_natural_order.csv
MKB10.csv	 RTL_Wiki.csv


In [8]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/ruwiki_good.txt',
)

dataset.get_possible_modalities()

{'@categories', '@lemmatized', '@ngramms'}

In [9]:
MAIN_MODALITY = '@lemmatized'

In [10]:
dataset._data.head()

,vw_text,raw_text,id
id,,,
Санкт-Петербург,Санкт-Петербург |@lemmatized год:301 петроград...,,Санкт-Петербург
Дворцовая_площадь,Дворцовая_площадь |@lemmatized дворцовый:43 пл...,,Дворцовая_площадь
Греко-персидские_войны,Греко-персидские_войны |@lemmatized грёкий:23 ...,,Греко-персидские_войны
Тихий_океан,Тихий_океан |@lemmatized тихий:92 океан:174 ус...,,Тихий_океан
Атлантический_океан,Атлантический_океан |@lemmatized атлантический...,,Атлантический_океан


In [11]:
dataset.get_dictionary()

artm.Dictionary(name=b0b21306-de87-407f-a250-8021dd4d9367, num_entries=892938)

In [12]:
dictionary = dataset.get_dictionary()

In [13]:
print(dictionary)

for modality in dataset.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=b0b21306-de87-407f-a250-8021dd4d9367, num_entries=892938)


In [14]:
dictionary.filter(min_df=5, max_df_rate=0.5)

artm.Dictionary(name=b0b21306-de87-407f-a250-8021dd4d9367, num_entries=61688)

In [15]:
dataset._cached_dict = dictionary

In [16]:
dataset.get_dictionary()

artm.Dictionary(name=b0b21306-de87-407f-a250-8021dd4d9367, num_entries=61688)

In [17]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [18]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 49.3 s, sys: 1.3 s, total: 50.5 s
Wall time: 50 s


In [19]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [20]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [21]:
def view_model(
        topic_model,
        dataset,
        num_top_tokens: int = 5,
        top_tokens_method: str = 'phi',
        num_topics: Optional[int] = 5,  # we do not want to fill the whole .ipynb notebook with topics...
        ):
    top_tok_viewer = TopTokensViewer(
        topic_model, num_top_tokens=num_top_tokens, method=top_tokens_method
    )
    top_doc_viewer = TopDocumentsViewer(topic_model, dataset=dataset)
    top_docs = top_doc_viewer.view()

    if num_topics is None:
        num_topics = len(topic_model.topic_names)

    for topic_name in topic_model.topic_names[:num_topics]:
        topic_top_toks = top_tok_viewer.to_html(topic_names=[topic_name])
        topic_top_docs = top_docs[topic_name]
        display_html(topic_top_toks, raw=True)
        display(topic_top_docs)

In [22]:
class FastFixPhiRegularizer(BaseRegularizer):
    _VERY_BIG_TAU = 10 ** 12  # Changed here 10 ** 9 

    def __init__(self, name: str, parent_model, topic_names: List[str], parent_phi=None):
        super().__init__(name, tau=self._VERY_BIG_TAU)

        self._topic_names = topic_names
        self._topic_indices = None
        self._parent_model = parent_model
        self._parent_phi = parent_phi

    def grad(self, pwt, nwt):
        # print('Fixing')

        rwt = np.zeros_like(pwt)

        if self._parent_phi is not None:
            parent_phi = parent_phi
        else:
            parent_phi = self._parent_model.get_phi()
        
        rwt[:, self._topic_indices] += parent_phi.values[:, self._topic_indices]

        return self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = self._model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

In [23]:
class DecorrelatorWithOtherPhiRegularizer(BaseRegularizer):
    def __init__(self, name, tau, topic_names, other_phi):
        super().__init__(name, tau=tau)

        self._topic_names = topic_names
        self._other_phi = other_phi
        self._other_topic_sum = self._other_phi.values.sum(
            axis=1, keepdims=True
        )
        
        self._topic_indices = None
        
    def grad(self, pwt, nwt):
        # print('Decorring')

        rwt = np.zeros_like(pwt)
        rwt[:, self._topic_indices] += (
            pwt.values[:, self._topic_indices] * self._other_topic_sum
        )

        return -1 * self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

In [24]:
class DecorrelatorWithOtherPhiRegularizer2(BaseRegularizer):
    def __init__(self, name, tau, topic_names, other_phi, num_iters: Optional[int] = None):
        super().__init__(name, tau=tau)

        self._topic_names = topic_names
        self._other_phi = other_phi
        self._num_iters = num_iters
        self._cur_iter = 0
        
        self._topic_indices = None
        
    def grad(self, pwt, nwt):
        rwt = np.zeros_like(pwt)
        
        if self._num_iters is not None and self._cur_iter >= self._num_iters:
            return rwt

        correlations = cdist(
            self._other_phi.values.T,
            pwt.values[:, self._topic_indices].T,
            lambda u, v: (u * v).sum()
        )
        weighted_other_topics = self._other_phi.values.dot(correlations)

        rwt[:, self._topic_indices] += (
            pwt.values[:, self._topic_indices] * weighted_other_topics
        )
        self._cur_iter += 1

        return -1 * self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

In [25]:
NUM_TOPICS = 20  # vary
MAX_NUM_TRAINS = 20
NUM_ITERATIONS = 10
NUM_TOP_TOKENS = 20

In [26]:
def fit_and_compute_scores(model, dataset, custom_regularizers=None):
    print(custom_regularizers)

    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS, custom_regularizers=custom_regularizers)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account
    target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}

    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
    }

In [27]:
def is_good(coherence):
    # 80 p
    return 0.8632394176407866 <= coherence

def is_bad(coherence):
    # 20 p
    return coherence <= 0.4889507901131518

## Test

In [33]:
model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=NUM_TOPICS,
    seed=2024,
)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



In [30]:
result = fit_and_compute_scores(model, dataset)

None


In [32]:
result['topic_coherences']

{0: 0.5904090207269129,
 1: 1.3298810340740868,
 2: 0.8532744272810638,
 3: 0.754056124738495,
 4: 0.5159992186033836,
 5: 0.8635976617111828,
 6: 0.46252709108303014,
 7: 0.4421710642640724,
 8: 0.823987698878097,
 9: 0.5702930794810561,
 10: 0.6679612961608734,
 11: 0.46099762215151907,
 12: 0.6321300507683978,
 13: 0.8187013124326967,
 14: 0.5436572568484732,
 15: 1.0132670176603482,
 16: 0.4418772183617042,
 17: 0.8553852728797514,
 18: 1.0858839256996495,
 19: 0.8409145909426826}

In [101]:
good_topic_indices = [
    t for t, c in result['topic_coherences'].items() if is_good(c)  # c >= HIGH_COHERENCE_THRESHOLD
]
bad_topic_indices = [
    t for t, c in result['topic_coherences'].items() if is_bad(c)  # c <= LOW_COHERENCE_THRESHOLD
]

phi = model.get_phi()
good_topic_names = [phi.columns[t] for t in good_topic_indices]
bad_topic_names = [phi.columns[t] for t in bad_topic_indices]

In [102]:
len(good_topic_indices), len(bad_topic_indices)

(3, 4)

In [103]:
good_topic_names, bad_topic_names

(['topic_3', 'topic_11', 'topic_15'],
 ['topic_8', 'topic_13', 'topic_18', 'topic_19'])

In [99]:
phi['topic_11'].sort_values(ascending=False)[:20]

modality  token       
@word     россия          0.008695
          война           0.008296
          государство     0.008275
          власть          0.007372
          страна          0.006107
          германия        0.005124
          политический    0.005033
          сталин          0.004779
          стать           0.004387
          революция       0.004353
          политика        0.003816
          франция         0.003641
          партия          0.003525
          военный         0.003510
          сторона         0.003133
          русский         0.003102
          должный         0.003097
          демократия      0.002889
          вопрос          0.002860
          народ           0.002849
Name: topic_11, dtype: float32

In [106]:
fix_regularizer = FastFixPhiRegularizer(
    name='fix',
    parent_model=model._model,
    topic_names=good_topic_names,
)

other_phi = model._model.get_phi()[bad_topic_names]
other_phi = deepcopy(other_phi)
decorr_bad_regularizer = DecorrelatorWithOtherPhiRegularizer(
    name='ext_decorr_bad', tau=25,  # 1e5
    topic_names=bad_topic_names,
    other_phi=other_phi
)

other_phi = model._model.get_phi()[good_topic_names]
other_phi = deepcopy(other_phi)
decorr_good_regularizer = DecorrelatorWithOtherPhiRegularizer(
    name='ext_decorr_good', tau=25,  # 1e5
    topic_names=good_topic_names,
    other_phi=other_phi
)

In [107]:
%%time

model._fit(
    dataset.get_batch_vectorizer(),
    num_iterations=10,
    custom_regularizers={
        fix_regularizer.name: fix_regularizer,
        decorr_bad_regularizer.name: decorr_bad_regularizer,
        decorr_good_regularizer.name: decorr_good_regularizer,
    }
)

CPU times: user 19.9 s, sys: 0 ns, total: 19.9 s
Wall time: 10.9 s


In [108]:
other_phi = model._model.get_phi()[bad_topic_names]
other_phi = deepcopy(other_phi)

In [112]:
other_phi.rename(
    columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
)

In [116]:
pd.concat([other_phi, model._model.get_phi(['topic_0'])], axis=1)

,m1_topic_8,m1_topic_13,m1_topic_18,m1_topic_19,topic_0
инвалидность,7.263344e-06,0.000000e+00,0.000000e+00,0.000000e+00,1.535188e-09
мазка,0.000000e+00,0.000000e+00,2.024645e-05,1.530354e-05,9.608340e-14
professor,0.000000e+00,3.533544e-13,1.517497e-12,0.000000e+00,0.000000e+00
умно,1.804371e-11,0.000000e+00,2.047675e-05,1.227452e-15,0.000000e+00
игил,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
...,...,...,...,...,...
милосердие,1.776537e-05,0.000000e+00,3.757288e-16,2.070272e-14,2.191762e-05
поверка,0.000000e+00,6.251613e-06,2.226501e-05,0.000000e+00,3.302222e-06
вто,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
слоить,1.872259e-05,6.039159e-16,3.578878e-05,0.000000e+00,1.846761e-15


In [28]:
def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [29]:
NUM_TRAINS = 3
TOPIC_INDICES = list(range(NUM_TOPICS))

In [30]:
TOPIC_INDICES

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

In [31]:
prev_results = dict()
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer
# DECORRELATION_TAUS = [10, 100, 1000, 10000, 1e5, 1e6, 1e7]
# DECORRELATION_TAUS = [1e8, 1e9, 1e10]
DECORRELATION_TAUS = [1e5, 1e6, 1e7, 1e8]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    prev_results[key] = []
    results[key] = []

    print(key)

    for seed in range(NUM_TRAINS):
        print(seed)
        
        model = init_model_from_family(
            family=KnownModel.ARTM,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params={
                'decorrelation_tau': 0.01,  # best values
                'smooth_bcg_tau': 0.05,
                'sparse_sp_tau': -0.05,
            }
        )

        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

        result = fit_and_compute_scores(model, dataset)
        prev_results[key].append(result)

        
        good_topic_indices = [
            t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
        ]
        bad_topic_indices = [
            t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
        ]
        not_good_topic_indices = [
            t for t in TOPIC_INDICES if t not in good_topic_indices
        ]
        
        phi = model.get_phi()
        good_topic_names = [phi.columns[t] for t in good_topic_indices]
        bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
        not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]


        assert len(good_topic_names) > 0
        assert len(bad_topic_names) > 0

        print(len(good_topic_names), len(bad_topic_names), len(not_good_topic_names))


        fix_regularizer = FastFixPhiRegularizer(
            name='fix',
            parent_model=model._model,
            topic_names=good_topic_names,
        )
        
        bad_phi = model._model.get_phi()[bad_topic_names]
        bad_phi = deepcopy(bad_phi)
        decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
            name='ext_decorr_bad', tau=decorrelation_tau,
            topic_names=not_good_topic_names,
            other_phi=bad_phi
        )
        
        good_phi = model._model.get_phi()[good_topic_names]
        good_phi = deepcopy(good_phi)
        decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
            name='ext_decorr_good', tau=decorrelation_tau,
            topic_names=not_good_topic_names,
            other_phi=good_phi
        )

        
        assert not hasattr(fix_regularizer, '_model')


        new_model = init_model_from_family(
            family=KnownModel.ARTM,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            specific_topic_names=not_good_topic_names,
            model_params={
                'decorrelation_tau': 0.01,
                'smooth_bcg_tau': 0.05,
                'sparse_sp_tau': -0.05,
            }
        )
        custom_regularizers = {
            fix_regularizer.name: fix_regularizer,
            decorr_bad_regularizer.name: decorr_bad_regularizer,
            decorr_good_regularizer.name: decorr_good_regularizer,
        }

        new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
        
        for reg in new_model.regularizers.data:
            print(f"{reg}: {new_model.regularizers[reg].tau}")

        for reg_name, reg in custom_regularizers.items():
            print(f"{reg_name}: {reg.tau}")

        
        results[key].append(new_result)

100000.0
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.4713878133278989
sparse_theta_sp: -3.380096644027831
decorrelation: 0.01
None
5 3 15


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7fe124867df0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7fe124867ee0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7fe124867d30>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.6285170844371986
sparse_theta_sp: -4.506795525370441
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 100000.0
ext_decorr_good: 100000.0
1
No spec topics
No spec topics


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7fe107240f70>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7fe107240ca0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7fe1075f7070>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.6734111618969985
sparse_theta_sp: -4.828709491468329
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 100000000.0
ext_decorr_good: 100000000.0
2
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.4713878133278989
sparse_theta_sp: -3.380096644027831
decorrelation: 0.01
None
5 4 15


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7fe11b9281f0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7fe11bec0580>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7fe11beaa400>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.6285170844371986
sparse_theta_sp: -4.506795525370441
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 100000000.0
ext_decorr_good: 100000000.0


In [ ]:
for k, r in prev_results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

In [ ]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

In [ ]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)

    prev_r = prev_results[k]
    prev_mean_ppl = sum(v['scores']['perplexity'] for v in prev_r) / len(prev_r)

    print(k, mean_ppl - prev_mean_ppl)

In [ ]:
#  Best: 1000000.0 32.44954427083394
# Close: 10000000.0 48.14615885416697
# Edgy:  100000000.0 761.1513671875

In [35]:
prev_results = dict()
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer2
# DECORRELATION_TAUS = [10, 100, 1000, 10000, 1e5, 1e6, 1e7, 1e8, 1e9, 1e10]
# DECORRELATION_TAUS = [1e11, 1e12, 1e13]
DECORRELATION_TAUS = [1e9, 1e10, 1e11, 1e12]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    prev_results[key] = []
    results[key] = []

    print(key)

    for seed in range(NUM_TRAINS):
        print(seed)
        
        model = init_model_from_family(
            family=KnownModel.ARTM,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params={
                'decorrelation_tau': 0.01,  # best values
                'smooth_bcg_tau': 0.05,
                'sparse_sp_tau': -0.05,
            }
        )

        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

        result = fit_and_compute_scores(model, dataset)
        prev_results[key].append(result)

        
        good_topic_indices = [
            t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
        ]
        bad_topic_indices = [
            t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
        ]
        not_good_topic_indices = [
            t for t in TOPIC_INDICES if t not in good_topic_indices
        ]
        
        phi = model.get_phi()
        good_topic_names = [phi.columns[t] for t in good_topic_indices]
        bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
        not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]


        assert len(good_topic_names) > 0
        assert len(bad_topic_names) > 0

        print(len(good_topic_names), len(bad_topic_names), len(not_good_topic_names))


        fix_regularizer = FastFixPhiRegularizer(
            name='fix',
            parent_model=model._model,
            topic_names=good_topic_names,
        )
        
        bad_phi = model._model.get_phi()[bad_topic_names]
        bad_phi = deepcopy(bad_phi)
        decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
            name='ext_decorr_bad', tau=decorrelation_tau,
            topic_names=not_good_topic_names,
            other_phi=bad_phi
        )
        
        good_phi = model._model.get_phi()[good_topic_names]
        good_phi = deepcopy(good_phi)
        decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
            name='ext_decorr_good', tau=decorrelation_tau,
            topic_names=not_good_topic_names,
            other_phi=good_phi
        )

        
        assert not hasattr(fix_regularizer, '_model')


        new_model = init_model_from_family(
            family=KnownModel.ARTM,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            specific_topic_names=not_good_topic_names,
            model_params={
                'decorrelation_tau': 0.01,
                'smooth_bcg_tau': 0.05,
                'sparse_sp_tau': -0.05,
            }
        )
        custom_regularizers = {
            fix_regularizer.name: fix_regularizer,
            decorr_bad_regularizer.name: decorr_bad_regularizer,
            decorr_good_regularizer.name: decorr_good_regularizer,
        }

        new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
        
        for reg in new_model.regularizers.data:
            print(f"{reg}: {new_model.regularizers[reg].tau}")

        for reg_name, reg in custom_regularizers.items():
            print(f"{reg_name}: {reg.tau}")

        
        results[key].append(new_result)

1000000000.0
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.4713878133278989
sparse_theta_sp: -3.380096644027831
decorrelation: 0.01
None
5 3 15


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7fe1026f2eb0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7fe1073f4ee0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7fe10715e2e0>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.6285170844371986
sparse_theta_sp: -4.506795525370441
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 1000000000.0
ext_decorr_good: 1000000000.0
1
No spec topics
No spec topics


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7fe11beaadc0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7fe102cb5bb0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7fe1248bbe50>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.6734111618969985
sparse_theta_sp: -4.828709491468329
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 1000000000000.0
ext_decorr_good: 1000000000000.0
2
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.4713878133278989
sparse_theta_sp: -3.380096644027831
decorrelation: 0.01
None
5 4 15


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7fe119beaa30>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7fe060dada90>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7fe1077ae490>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.6285170844371986
sparse_theta_sp: -4.506795525370441
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 1000000000000.0
ext_decorr_good: 1000000000000.0


In [36]:
for k, r in prev_results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

1000000000.0 4876.149739583333
10000000000.0 4876.149739583333
100000000000.0 4876.149739583333
1000000000000.0 4876.14990234375


In [37]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

1000000000.0 4909.190266927083
10000000000.0 4914.046712239583
100000000000.0 5069.98388671875
1000000000000.0 5668.677408854167


In [38]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)

    prev_r = prev_results[k]
    prev_mean_ppl = sum(v['scores']['perplexity'] for v in prev_r) / len(prev_r)

    print(k, mean_ppl - prev_mean_ppl)

1000000000.0 33.04052734375
10000000000.0 37.89697265625
100000000000.0 193.83414713541697
1000000000000.0 792.527506510417


In [56]:
#  Best: 1000000000.0 33.04052734375
# Close: 10000000000.0 37.89697265625
# Edgy: 100000000000.0 193.83414713541697

In [57]:
MAX_NUM_TRAINS

20

In [58]:
results

{100000000000.0: [{'scores': {'perplexity': 5060.9853515625,
    'coherence_20': array([0.88851644]),
    'diversity_euclidean': 0.05181928580608785,
    'diversity_jensenshannon': 0.6892597676864831,
    'diversity_hellinger': 0.8005273281737059,
    'diversity_cosine': 0.8730561673943762},
   'topic_coherences': {0: 0.9510485130620158,
    1: 0.8244207819413056,
    2: 0.6532960615322192,
    3: 0.8791889755098058,
    4: 1.2769516166172947,
    5: 0.6605213956839725,
    6: 1.0297691355246918,
    7: 0.6100542371115539,
    8: 0.6882731440229847,
    9: 1.1756616093491055,
    10: 0.7716390432284849,
    11: 1.3298810340740868,
    12: 0.9104903714039343,
    13: 1.0742719078646432,
    14: 0.7752013512956494,
    15: 0.9857446836345978,
    16: 0.6336138082063062,
    17: 0.750068974495246,
    18: 1.058618543631895,
    19: 0.7316135662705701}},
  {'scores': {'perplexity': 5109.7724609375,
    'coherence_20': array([0.94364784]),
    'diversity_euclidean': 0.05511951253098467,
   

In [43]:
new_result

{'scores': {'perplexity': 2461.866943359375,
  'coherence_20': array([1.85850209]),
  'diversity_euclidean': 0.07356362067834214,
  'diversity_jensenshannon': 0.7711511659469856,
  'diversity_hellinger': 0.9155164493873434,
  'diversity_cosine': 0.9444821285586293},
 'topic_coherences': {0: 1.7300523146588405,
  1: 1.5709401517076538,
  2: 1.9563085816159047,
  3: 1.6460933884449667,
  4: 2.26887264809252,
  5: 2.133835008205159,
  6: 2.107893087858345,
  7: 1.6728566636808413,
  8: 2.237799559639728,
  9: 1.8893206007995285,
  10: 1.9824795351569895,
  11: 2.159035393982212,
  12: 0.9739755322110191,
  13: 2.2229610002548172,
  14: 1.2609566862135837,
  15: 1.6770159675711767,
  16: 2.8450990542764627,
  17: 1.6517525523607473,
  18: 2.116133177601322,
  19: 1.066660946339064}}

In [44]:
fix_regularizer._topic_names

['topic_0',
 'topic_2',
 'topic_3',
 'topic_8',
 'topic_10',
 'topic_11',
 'topic_16']

In [36]:
del model

NameError: name 'model' is not defined

In [32]:
NUM_ITERATIONS

10

In [31]:
NUM_GOOD_TOPICS_THRESHOLD = NUM_TOPICS - 2

In [32]:
import json

SAVE_FOLDER = 'results/ruwikigood'

! mkdir -p $SAVE_FOLDER

In [34]:
! ls $SAVE_FOLDER

decorrelation.json	  _iterative_1000000.json  plsa.json	tless.json
_iterative_10000000.json  lda.json		   sparse.json


In [35]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer

#  Best: 1000000.0 32.46110026041697
# Close: 10000000.0 48.17822265625
# Edgy:  100000000.0 761.118815104167

DECORRELATION_TAUS = [1000000, 10000000, 100000000]
# DECORRELATION_TAUS = [10000000, 100000000]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        seed_save_folder = os.path.join(SAVE_FOLDER, f'iterative_{key}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'iterative_{key}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            raise NotImplementedError()

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1
            
            del result, phi
            model = None

        else:

            fix_regularizer = FastFixPhiRegularizer(
                name='fix',
                parent_model=prev_model._model,
                topic_names=good_topic_names,
            )
            
            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                decorr_good_regularizer.name: decorr_good_regularizer,
            }
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

            
            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            # TODO: rm
            

    
    for k, r in results.items():
        for s in r:
            s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
        
    for k, r in results.items():
        with open(SAVE_FOLDER + f'/iterative_{int(k)}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

1000000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.4713878133278989
sparse_theta_sp: -3.380096644027831
decorrelation: 0.01
None
num_topics: {'good': 5, 'bad': 3, 'not_good': 15, 'total_bad': 3}
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f54688d9d00>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f54688d9fd0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f54688d9ee0>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.6285170844371986
sparse_theta_sp: -4.506795525370441
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 1000000
ext_decorr_good: 1000000
SUCCESS: less bad topics
num_topics: {'good': 5, 'bad': 2, 'not_good': 15, 'total_bad': 5}
2


...

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f5484232700>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f5488a35640>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f5484232250>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -2.356939066639495
sparse_theta_sp: -16.900483220139154
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
DOWNFALL: more bad topics...
num_topics: {'good': 16, 'bad': 4, 'not_good': 4, 'total_bad': 24}
8


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f54842a7a90>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f5484232940>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f548d073070>}


AssertionError: (136, 190)

In [36]:
1

1

In [37]:
results.keys()

dict_keys([1000000, 10000000, 100000000])

In [38]:
! ls $SAVE_FOLDER

decorrelation.json   _iterative_10000000.json  lda.json
iterative_1000000    iterative_10000000.json   plsa.json
iterative_10000000   _iterative_1000000.json   sparse.json
iterative_100000000  iterative_1000000.json    tless.json


In [64]:
phi.T.shape

(21, 61688)

In [65]:
from scipy.spatial.distance import pdist

phi = new_model.get_phi()
phi = phi.iloc[:, :-1]

for metric in KNOWN_METRICS:
    condensed_distances = pdist(phi.T, metric=metric)
    print(condensed_distances.shape)

(190,)
(190,)


ValueError: Unknown Distance Metric: hellinger

In [61]:
NUM_TOPICS

20

In [58]:
diversity_scores = [
DiversityScore(
    name=f'diversity_{metric}',
) for metric in KNOWN_METRICS
]
for score in diversity_scores:
    value = score.call(new_model)

In [57]:
len(results[10000000])

14

In [39]:
len(good_topic_names), good_topic_names

(12,
 ['topic_1',
  'topic_2',
  'topic_3',
  'topic_5',
  'topic_6',
  'topic_8',
  'topic_9',
  'topic_11',
  'topic_12',
  'topic_13',
  'topic_15',
  'topic_17'])

In [40]:
len(new_good_topic_names), new_good_topic_names

(14,
 ['topic_0',
  'topic_1',
  'topic_2',
  'topic_5',
  'topic_6',
  'topic_7',
  'topic_8',
  'topic_9',
  'topic_11',
  'topic_12',
  'topic_13',
  'topic_14',
  'topic_15',
  'topic_17'])

In [41]:
results.keys()

dict_keys([10000000])

In [42]:
results[10000000][-2]

{'scores': {'perplexity': 5028.8447265625,
  'coherence_20': array([0.86319712]),
  'diversity_euclidean': 0.054818420459782365,
  'diversity_jensenshannon': 0.6949776823406634,
  'diversity_hellinger': 0.8081380542846611,
  'diversity_cosine': 0.8821286852315896},
 'topic_coherences': {0: 0.8247326578363814,
  1: 1.0187119263886386,
  2: 1.0314460708720639,
  3: 0.9020839792488323,
  4: 0.6693972945135833,
  5: 1.0132594869592875,
  6: 1.0297691355246918,
  7: 0.5952443364480671,
  8: 0.9947305549797112,
  9: 1.1756616093491055,
  10: 0.5864853710832567,
  11: 1.3298810340740868,
  12: 0.9104903714039343,
  13: 0.8776684805879968,
  14: 0.4776456805576165,
  15: 0.9857446836345978,
  16: 0.5762867659181736,
  17: 0.8855684065852913,
  18: 0.7549591663381537,
  19: 0.6241753707184994},
 'num_topics': {'good': 12, 'bad': 1, 'not_good': 8, 'total_bad': 5}}

In [43]:
results[10000000][-1]

{'scores': {'perplexity': 5157.0244140625,
  'coherence_20': array([0.92829873]),
  'diversity_euclidean': 0.06093881509567182,
  'diversity_jensenshannon': 0.7174395169305123,
  'diversity_hellinger': 0.8385603728567432,
  'diversity_cosine': 0.9097192918544621},
 'topic_coherences': {0: 1.0964611080106725,
  1: 1.0187119263886386,
  2: 1.0314460708720639,
  3: 0.7987289859960877,
  4: 0.8101728099102101,
  5: 1.0132594869592875,
  6: 1.0297691355246918,
  7: 0.8732806734879869,
  8: 0.9947305549797112,
  9: 1.1756616093491055,
  10: 0.7979896218267979,
  11: 1.3298810340740868,
  12: 0.9104903714039343,
  13: 0.8776684805879968,
  14: 1.1999101759379773,
  15: 0.9857446836345978,
  16: 0.506286909860512,
  17: 0.8855684065852913,
  18: 0.5274013676211138,
  19: 0.702811229144044}}

In [50]:
prev_model.get_phi()['topic_3'].sort_values(ascending=False)[:21]

modality     token       
@lemmatized  консул          0.014332
             рим             0.013379
             язык            0.012402
             цезарь          0.011830
             римский         0.010164
             квинта          0.009347
             сенат           0.008172
             сципион         0.006341
             марка           0.005726
             алфавит         0.005091
             говор           0.004926
             слово           0.004922
             источник        0.004488
             политический    0.004387
             диалект         0.004376
             провинция       0.004222
             род             0.004188
             народный        0.003656
             красс           0.003257
             трибуна         0.003064
             форма           0.003063
Name: topic_3, dtype: float32

In [51]:
new_model.get_phi()['topic_3'].sort_values(ascending=False)[:21]

modality     token       
@lemmatized  консул          0.014331
             рим             0.013380
             язык            0.012405
             цезарь          0.011829
             римский         0.010165
             квинта          0.009346
             сенат           0.008171
             сципион         0.006341
             марка           0.005726
             алфавит         0.005091
             говор           0.004925
             слово           0.004922
             источник        0.004489
             политический    0.004387
             диалект         0.004376
             провинция       0.004222
             род             0.004189
             народный        0.003656
             красс           0.003257
             форма           0.003064
             трибуна         0.003063
Name: topic_3, dtype: float32

In [54]:
# TODO: we see that two last words (20, 21) swapped places --> coherence become worse
# trying to increase tau for fix (10 ** 12)
# or increase num iters?... (but it won't be fair, because all other trained with 10 iters)

In [92]:
set(good_topic_names) <= set(new_good_topic_names)

True

In [91]:
new_good_topic_names

['topic_0',
 'topic_1',
 'topic_2',
 'topic_3',
 'topic_4',
 'topic_5',
 'topic_6',
 'topic_7',
 'topic_8',
 'topic_9',
 'topic_11',
 'topic_12',
 'topic_13',
 'topic_14',
 'topic_15',
 'topic_16',
 'topic_17',
 'topic_19']

In [67]:
decorrelation_tau

10000000

DECORR_TAU = 10000000

```
--> 176 assert set(good_topic_names) <= set(new_good_topic_names)
    177 # assert len(new_bad_topic_names) <= len(bad_topic_names)
    179 if len(new_good_topic_names) > len(good_topic_names):

AssertionError: 
```


DECORR_TAU = 10000000

File ~/projects/iterative/../OptimalNumberOfTopics/topnum/scores/diversity_score.py:159, in _DiversityScore.call(self, model)
    157 condensed_distances = condensed_distances[np.isfinite(condensed_distances)]
    158 filtered_num_dists = len(condensed_distances)
--> 159 assert filtered_num_dists >= 0.9 * orig_num_dists, (filtered_num_dists, orig_num_dists)
    161 if self.closest:
    162     df = pd.DataFrame(
    163         index=phi.columns, columns=phi.columns,
    164         data=squareform(condensed_distances)
    165     )

AssertionError: (153, 190)

In [37]:
decorrelation_tau # We skip it (again error)

100000000

-> 159 assert filtered_num_dists >= 0.9 * orig_num_dists, (filtered_num_dists, orig_num_dists)
    161 if self.closest:
    162     df = pd.DataFrame(
    163         index=phi.columns, columns=phi.columns,
    164         data=squareform(condensed_distances)
    165     )

AssertionError: (136, 190)

In [39]:
results.keys()

dict_keys([1000000, 10000000, 100000000])

In [40]:
for k, r in results.items():
    for s in r:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [ ]:
import json

SAVE_FOLDER = 'results/ruwikigood'

! mkdir -p $SAVE_FOLDER

In [41]:
for k, r in results.items():
    with open(SAVE_FOLDER + f'/iterative_{int(k)}.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )

In [61]:
! ls $SAVE_FOLDER

decorrelation.json		     iterative2_10000000000
iterative_1000000		     iterative2_100000000000
iterative_10000000		     iterative2_100000000000.json
iterative_100000000		     iterative2_10000000000.json
iterative_100000000_unfinished.json  iterative2_1000000000.json
_iterative_10000000.json	     lda.json
iterative_10000000.json		     plsa.json
_iterative_1000000.json		     sparse.json
iterative_1000000.json		     tless.json
iterative2_1000000000


In [49]:
! tail -n 50 $SAVE_FOLDER/iterative_1000000.json

            "17": 0.8860349099275922,
            "18": 0.915649713332233,
            "19": 0.9205320040732358
        },
        "num_topics": {
            "good": 17,
            "bad": 0,
            "not_good": 3,
            "total_bad": 28
        }
    },
    {
        "scores": {
            "perplexity": 5484.72119140625,
            "coherence_20": 1.0236179923056614,
            "diversity_euclidean": 0.09214386728357007,
            "diversity_jensenshannon": 0.7376052118185588,
            "diversity_hellinger": 0.8663442123491949,
            "diversity_cosine": 0.9288833090486716
        },
        "topic_coherences": {
            "0": 1.1117905194294226,
            "1": 1.0073674906547048,
            "2": 0.9541544081404468,
            "3": 1.0849412776146068,
            "4": 1.4883995284774352,
            "5": 0.891633574241691,
            "6": 1.0297691355246918,
            "7": 0.8975712771845624,
            "8": 0.9395098499846037,
            "9": 1.1756

In [47]:
! mv $SAVE_FOLDER/iterative_100000000.json $SAVE_FOLDER/iterative_100000000_unfinished.json

In [70]:
! tail -n 50 $SAVE_FOLDER/iterative_10000000.json

            "11": 1.3298810340740868,
            "12": 0.9104903714039343,
            "13": 0.8776684805879968,
            "14": 0.4776456805576165,
            "15": 0.9857446836345978,
            "16": 0.5762867659181736,
            "17": 0.8855684065852913,
            "18": 0.7549591663381537,
            "19": 0.6241753707184994
        },
        "num_topics": {
            "good": 12,
            "bad": 1,
            "not_good": 8,
            "total_bad": 5
        }
    },
    {
        "scores": {
            "perplexity": 5157.02392578125,
            "coherence_20": 0.9282987321077405,
            "diversity_euclidean": 0.06093881333161304,
            "diversity_jensenshannon": 0.7174395173674312,
            "diversity_hellinger": 0.8385603701792002,
            "diversity_cosine": 0.9097192934154914
        },
        "topic_coherences": {
            "0": 1.0964611080106725,
            "1": 1.0187119263886386,
            "2": 1.0314460708720639,
            "3":

In [ ]:
results.keys()

In [50]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer2

#  Best: 1000000000.0 33.05615234375
# Close: 10000000000.0 37.94612630208394
# Edgy: 100000000000.0 193.84537760416697

DECORRELATION_TAUS = [1000000000, 10000000000, 100000000000]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        seed_save_folder = os.path.join(SAVE_FOLDER, f'iterative2_{key}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'iterative2_{key}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            raise NotImplementedError()

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1
            
            del result, phi
            model = None

        else:

            fix_regularizer = FastFixPhiRegularizer(
                name='fix',
                parent_model=prev_model._model,
                topic_names=good_topic_names,
            )
            
            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                decorr_good_regularizer.name: decorr_good_regularizer,
            }
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

            
            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            # TODO: rm
            

    
    for k, r in results.items():
        for s in r:
            s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
        
    for k, r in results.items():
        with open(SAVE_FOLDER + f'/iterative2_{int(k)}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

1000000000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.4713878133278989
sparse_theta_sp: -3.380096644027831
decorrelation: 0.01
None
num_topics: {'good': 5, 'bad': 3, 'not_good': 15, 'total_bad': 3}
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f5484584700>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f5481fea130>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f547f69f1f0>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.6285170844371986
sparse_theta_sp: -4.506795525370441
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 1000000000
ext_decorr_good: 1000000000
SUCCESS: less bad topics
num_topics: {'good': 5, 'bad': 2, 'not_good': 15, 'total_bad': 5}
2


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f547f77f5e0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f548d5b8c10>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f548d5b8ca0>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.6285170844371986
sparse_theta_sp: -4.506795525370441
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 100000000000
ext_decorr_good: 100000000000
SUCCESS: more good topics
SUCCESS: less bad topics
SUCCESS: no bad topics!
num_topics: {'good': 11, 'bad': 0, 'not_good': 9, 'total_bad': 3}
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f549e466820>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f548d5b8430>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f5488f70af0>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -1.0475284740619977
sparse_theta_sp: -7.511325875617401
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 100000000000
ext_decorr_good: 100000000000
SUCCESS: more good topics
SUCCESS: no bad topics!
num_topics: {'good': 14, 'bad': 0, 'not_good': 6, 'total_bad': 3}
3


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f547f7b1b80>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f549e466790>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f549e466160>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -1.5712927110929964
sparse_theta_sp: -11.266988813426103
decorrelation: 0.01
fix: 1000000000000
ext_decorr_bad: 100000000000
ext_decorr_good: 100000000000
SUCCESS: more good topics
SUCCESS: no bad topics!
num_topics: {'good': 19, 'bad': 0, 'not_good': 1, 'total_bad': 3}


In [51]:
1

1

In [52]:
results.keys()

dict_keys([1000000000, 10000000000, 100000000000])

In [ ]:
for k, r in results.items():
    for s in r:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [ ]:
1

In [55]:
for k, r in results.items():
    with open(SAVE_FOLDER + f'/iterative2_{int(k)}.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )

In [56]:
! ls $SAVE_FOLDER

decorrelation.json		     iterative2_10000000000
iterative_1000000		     iterative2_100000000000
iterative_10000000		     iterative2_100000000000.json
iterative_100000000		     iterative2_10000000000.json
iterative_100000000_unfinished.json  iterative2_1000000000.json
_iterative_10000000.json	     lda.json
iterative_10000000.json		     plsa.json
_iterative_1000000.json		     sparse.json
iterative_1000000.json		     tless.json
iterative2_1000000000


In [ ]:
view_model(prev_model, dataset)

In [59]:
! tail -n 50 $SAVE_FOLDER/iterative2_100000000000.json

            "17": 1.0945685662825082,
            "18": 1.6449581056412337,
            "19": 0.748622775554295
        },
        "num_topics": {
            "good": 14,
            "bad": 0,
            "not_good": 6,
            "total_bad": 3
        }
    },
    {
        "scores": {
            "perplexity": 5516.505859375,
            "coherence_20": 1.1268811326997057,
            "diversity_euclidean": 0.07014006156266861,
            "diversity_jensenshannon": 0.7581462981301444,
            "diversity_hellinger": 0.8944400092841264,
            "diversity_cosine": 0.9500833147005159
        },
        "topic_coherences": {
            "0": 0.9651943326806242,
            "1": 1.194845929143441,
            "2": 1.0945384001799925,
            "3": 0.8889544214070723,
            "4": 0.8455319740703664,
            "5": 1.5662820208527553,
            "6": 1.0297691355246918,
            "7": 0.9894365592181189,
            "8": 1.2937343856983365,
            "9": 1.1756616

In [60]:
! tail -n 50 $SAVE_FOLDER/iterative2_1000000.json

tail: cannot open 'results/ruwikigood/iterative2_1000000.json' for reading: No such file or directory


In [98]:
! tail -n 50 $SAVE_FOLDER/iterative2_1000000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.640629444035244
        },
        "num_topics": {
            "good": 13,
            "bad": 4,
            "not_good": 7,
            "total_bad": 54
        }
    },
    {
        "scores": {
            "perplexity": 2399.75244140625,
            "coherence_20": 1.5422349498469412,
            "diversity_euclidean": 0.07473976332049242,
            "diversity_jensenshannon": 0.7098907759986864,
            "diversity_hellinger": 0.8325292643052555,
            "diversity_cosine": 0.8483379385827017
        },
        "topic_coherences": {
            "0": 1.00113211617019,
            "1": 1.3621768174895768,
            "2": 1.6694630947055196,
            "3": 0.901739498209401,
            "4": 0.9697594159537758,
            "5": 0.6067957005801524,
            "6": 1.8054658007396136,
            "7": 0.907673304571028,
            "8": 0.9312208318641825,
            "9": 1.63002679

In [79]:
1

1

## Ablation Study

In [62]:
! ls $SAVE_FOLDER

decorrelation.json		     iterative2_10000000000
iterative_1000000		     iterative2_100000000000
iterative_10000000		     iterative2_100000000000.json
iterative_100000000		     iterative2_10000000000.json
iterative_100000000_unfinished.json  iterative2_1000000000.json
_iterative_10000000.json	     lda.json
iterative_10000000.json		     plsa.json
_iterative_1000000.json		     sparse.json
iterative_1000000.json		     tless.json
iterative2_1000000000


In [ ]:
# 1000000 10000000
# 10000000000 100000000000

In [33]:
! tail -n 50 $SAVE_FOLDER/iterative2_100000000000.json

            "17": 1.0945685662825082,
            "18": 1.6449581056412337,
            "19": 0.748622775554295
        },
        "num_topics": {
            "good": 14,
            "bad": 0,
            "not_good": 6,
            "total_bad": 3
        }
    },
    {
        "scores": {
            "perplexity": 5516.505859375,
            "coherence_20": 1.1268811326997057,
            "diversity_euclidean": 0.07014006156266861,
            "diversity_jensenshannon": 0.7581462981301444,
            "diversity_hellinger": 0.8944400092841264,
            "diversity_cosine": 0.9500833147005159
        },
        "topic_coherences": {
            "0": 0.9651943326806242,
            "1": 1.194845929143441,
            "2": 1.0945384001799925,
            "3": 0.8889544214070723,
            "4": 0.8455319740703664,
            "5": 1.5662820208527553,
            "6": 1.0297691355246918,
            "7": 0.9894365592181189,
            "8": 1.2937343856983365,
            "9": 1.1756616

In [106]:
! tail -n 50 $SAVE_FOLDER/iterative_1000000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.6457005218273515
        },
        "num_topics": {
            "good": 19,
            "bad": 0,
            "not_good": 1,
            "total_bad": 15
        }
    },
    {
        "scores": {
            "perplexity": 2509.45068359375,
            "coherence_20": 1.8343696152756745,
            "diversity_euclidean": 0.10217171248725132,
            "diversity_jensenshannon": 0.7695284135036325,
            "diversity_hellinger": 0.9123624284297017,
            "diversity_cosine": 0.9442764712790213
        },
        "topic_coherences": {
            "0": 0.6043036512713817,
            "1": 1.85624161842734,
            "2": 1.8789415533863691,
            "3": 2.0087784467994148,
            "4": 1.837319679382072,
            "5": 1.8946013416131207,
            "6": 1.90155438803718,
            "7": 1.6737924853770243,
            "8": 1.8444252589807613,
            "9": 2.11380410

In [70]:
DECORRELATION_TAUS = [1000000, 10000000]
DECORRELATION_TAUS2 = [10000000000, 100000000000, 1000000000]

In [34]:
! ls $SAVE_FOLDER/ablation_study

iterative_1000000_1-0-1.json


In [57]:
DECORRELATION_TAUS =  [1000000,     10000000]
DECORRELATION_TAUS2 = [10000000000, 100000000000, 1000000000]

DECORRELATION_TAU = 10000000  # 1000000

ALL_PARAMS = [
    # (0, 1, 1),
    (1, 0, 1),
    (1, 1, 0),

    (1, 0, 0),
    # (0, 1, 0),
    # (0, 0, 1),
]

In [ ]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer

for params in ALL_PARAMS:
    key = params

    output_k = '-'.join(str(i) for i in key)
    res_file_path = SAVE_FOLDER + f'/ablation_study/iterative_{DECORRELATION_TAU}_{output_k}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue

    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1
            
            del result, phi
            model = None

        else:
            custom_regularizers = dict()
            
            if params[0]:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_model=prev_model._model,
                    topic_names=good_topic_names,
                )
                custom_regularizers[fix_regularizer.name] = fix_regularizer
            else:
                fix_regularizer = None
            
            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)
    
            bad_phi = deepcopy(bad_phi)

            if params[1]:
                decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_bad', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=bad_phi
                )
                custom_regularizers[decorr_bad_regularizer.name] = decorr_bad_regularizer
            else:
                decorr_bad_regularizer = None
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)

            if params[2]:
                decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_good', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=good_phi
                )
                custom_regularizers[decorr_good_regularizer.name] = decorr_good_regularizer
            else:
                decorr_good_regularizer = None
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # assert len(new_good_topic_names) > 0
            if len(new_good_topic_names) == 0:
                print('DOWNFALL: no good topics...')
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # assert set(good_topic_names) <= set(new_good_topic_names)
            if not (set(good_topic_names) <= set(new_good_topic_names)):
                print('DOWNFALL: some good topics lost...')
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

    for k, r in results.items():
        for s in r:
            s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

    for k, r in results.items():
        output_k = '-'.join(str(i) for i in k)
        with open(SAVE_FOLDER + f'/ablation_study/iterative_{DECORRELATION_TAU}_{output_k}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

(1, 0, 1)
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.4713878133278989
sparse_theta_sp: -3.380096644027831
decorrelation: 0.01
None
num_topics: {'good': 5, 'bad': 3, 'not_good': 15, 'total_bad': 3}
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9c5741a100>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f9c6d6cd0a0>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.6285170844371986
sparse_theta_sp: -4.506795525370441
decorrelation: 0.01
fix: 1000000000000
ext_decorr_good: 10000000
SUCCESS: less bad topics
num_topics: {'good': 5, 'bad': 2, 'not_good': 15, 'total_bad': 5}
2


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9c6dab3820>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -1.0475284740619977
sparse_theta_sp: -7.511325875617401
decorrelation: 0.01
fix: 1000000000000
SUCCESS: less bad topics
num_topics: {'good': 11, 'bad': 3, 'not_good': 9, 'total_bad': 30}
11


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9c6de852e0>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -1.0475284740619977
sparse_theta_sp: -7.511325875617401
decorrelation: 0.01
fix: 1000000000000
SUCCESS: less bad topics
num_topics: {'good': 11, 'bad': 1, 'not_good': 9, 'total_bad': 31}
12


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9c34c160d0>}


In [59]:
1

1

In [72]:
SAVE_FOLDER

'results/ruwikigood'

In [73]:
! mkdir -p results/ruwikigood/ablation_study

In [51]:
results.keys()

dict_keys([(1, 0, 1)])

In [120]:
'-'.join(str(i) for i in k)

'0-0-1'

In [60]:
! ls results/ruwikigood/ablation_study

iterative_10000000_1-0-0.json  iterative_1000000_1-0-0.json
iterative_10000000_1-0-1.json  iterative_1000000_1-0-1.json
iterative_10000000_1-1-0.json  iterative_1000000_1-1-0.json


In [76]:
! tail -n 50 results/ruwikigood/ablation_study/iterative_10000000_1-1-0.json

            "17": 1.00759834586587,
            "18": 0.7894774428694977,
            "19": 0.8892402147895911
        },
        "num_topics": {
            "good": 17,
            "bad": 0,
            "not_good": 3,
            "total_bad": 9
        }
    },
    {
        "scores": {
            "perplexity": 5544.6298828125,
            "coherence_20": 0.9631467795815484,
            "diversity_euclidean": 0.14163592699090075,
            "diversity_jensenshannon": 0.7553822393498667,
            "diversity_hellinger": 0.8913671270392892,
            "diversity_cosine": 0.9368439177215284
        },
        "topic_coherences": {
            "0": 1.072046475970887,
            "1": 0.3513550413231771,
            "2": 0.9337644789515829,
            "3": 0.9751415508706324,
            "4": 0.9380578796823058,
            "5": 0.9005837729076939,
            "6": 1.0297691355246918,
            "7": 0.8975712771845623,
            "8": 0.9535938207656097,
            "9": 1.1756616

In [123]:
for k, r in results.items():
    print(k)
    print(r[-1]['scores'])
    print(r[-1]['num_topics'])
    print()

(0, 1, 1)
{'perplexity': 2298.86474609375, 'coherence_20': 1.7097599620715205, 'diversity_euclidean': 0.07696227654850847, 'diversity_jensenshannon': 0.75997374559108, 'diversity_hellinger': 0.8996516085590787, 'diversity_cosine': 0.8922071526632959}
{'good': 8, 'bad': 3, 'not_good': 12, 'total_bad': 68}

(1, 0, 1)
{'perplexity': 2464.59130859375, 'coherence_20': 1.8222803403386731, 'diversity_euclidean': 0.09665751707718807, 'diversity_jensenshannon': 0.7558104290820944, 'diversity_hellinger': 0.8931846991572905, 'diversity_cosine': 0.9294582819017179}
{'good': 19, 'bad': 1, 'not_good': 1, 'total_bad': 21}

(1, 1, 0)
{'perplexity': 2476.902099609375, 'coherence_20': 1.7674292440536103, 'diversity_euclidean': 0.09821210131545323, 'diversity_jensenshannon': 0.7519702519427854, 'diversity_hellinger': 0.8875734457352042, 'diversity_cosine': 0.9115068086898529}
{'good': 19, 'bad': 1, 'not_good': 1, 'total_bad': 12}

(1, 0, 0)
{'perplexity': 2399.752197265625, 'coherence_20': 1.532591468855

In [124]:
! ls $SAVE_FOLDER

ablation_study		iterative_100.json	    iterative2_100000.json
decorrelation.json	iterative2_1000000000.json  lda.json
iterative_1000000.json	iterative2_100000000.json   plsa.json
iterative_100000.json	iterative2_10000000.json    sparse.json
iterative_1000.json	iterative2_1000000.json     tless.json


In [107]:
! ls $SAVE_FOLDER

ablation_study		iterative_100.json	    iterative2_100000.json
decorrelation.json	iterative2_1000000000.json  lda.json
iterative_1000000.json	iterative2_100000000.json   plsa.json
iterative_100000.json	iterative2_10000000.json    sparse.json
iterative_1000.json	iterative2_1000000.json     tless.json


In [108]:
! tail -n 50 $SAVE_FOLDER/iterative2_1000000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.640629444035244
        },
        "num_topics": {
            "good": 13,
            "bad": 4,
            "not_good": 7,
            "total_bad": 54
        }
    },
    {
        "scores": {
            "perplexity": 2399.75244140625,
            "coherence_20": 1.5422349498469412,
            "diversity_euclidean": 0.07473976332049242,
            "diversity_jensenshannon": 0.7098907759986864,
            "diversity_hellinger": 0.8325292643052555,
            "diversity_cosine": 0.8483379385827017
        },
        "topic_coherences": {
            "0": 1.00113211617019,
            "1": 1.3621768174895768,
            "2": 1.6694630947055196,
            "3": 0.901739498209401,
            "4": 0.9697594159537758,
            "5": 0.6067957005801524,
            "6": 1.8054658007396136,
            "7": 0.907673304571028,
            "8": 0.9312208318641825,
            "9": 1.63002679

In [109]:
! tail -n 50 $SAVE_FOLDER/iterative2_10000000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.640629444035244
        },
        "num_topics": {
            "good": 13,
            "bad": 2,
            "not_good": 7,
            "total_bad": 45
        }
    },
    {
        "scores": {
            "perplexity": 2403.99267578125,
            "coherence_20": 1.573263970374759,
            "diversity_euclidean": 0.07375830564020532,
            "diversity_jensenshannon": 0.7124094332529214,
            "diversity_hellinger": 0.8359994991184351,
            "diversity_cosine": 0.855706460551422
        },
        "topic_coherences": {
            "0": 1.1082963221453308,
            "1": 1.4004677404363155,
            "2": 1.6549948867753843,
            "3": 0.9853736724876744,
            "4": 1.1166110279043346,
            "5": 0.6067957005801523,
            "6": 1.8054658007396136,
            "7": 1.0752934564635988,
            "8": 0.9390066126540663,
            "9": 1.630026

In [110]:
! tail -n 50 $SAVE_FOLDER/iterative2_100000000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.640629444035244
        },
        "num_topics": {
            "good": 19,
            "bad": 1,
            "not_good": 1,
            "total_bad": 17
        }
    },
    {
        "scores": {
            "perplexity": 2536.84521484375,
            "coherence_20": 1.7657066593419635,
            "diversity_euclidean": 0.08474110227762996,
            "diversity_jensenshannon": 0.7364641673959812,
            "diversity_hellinger": 0.8673032015098073,
            "diversity_cosine": 0.8908660854388641
        },
        "topic_coherences": {
            "0": 1.6603571660847984,
            "1": 1.8608120102679044,
            "2": 0.6321317616434708,
            "3": 2.08647789775495,
            "4": 1.8032283276088727,
            "5": 1.696860620500633,
            "6": 1.8054658007396138,
            "7": 1.651973818044545,
            "8": 1.7787379471054157,
            "9": 1.63002679

In [112]:
! tail -n 50 $SAVE_FOLDER/iterative2_1000000000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.6167513780664968
        },
        "num_topics": {
            "good": 19,
            "bad": 1,
            "not_good": 1,
            "total_bad": 14
        }
    },
    {
        "scores": {
            "perplexity": 2509.298095703125,
            "coherence_20": 1.8004192080579677,
            "diversity_euclidean": 0.09672882730245023,
            "diversity_jensenshannon": 0.7499680063404501,
            "diversity_hellinger": 0.8856458799229798,
            "diversity_cosine": 0.9193192437631968
        },
        "topic_coherences": {
            "0": 1.6834217755146534,
            "1": 1.6636143961405991,
            "2": 1.9126025217315148,
            "3": 1.7636570949116437,
            "4": 1.6862600158085381,
            "5": 1.9193798353133522,
            "6": 2.2642665670036526,
            "7": 0.6104632760028169,
            "8": 2.091593575026202,
            "9": 1.824

In [88]:
DECORRELATION_TAUS =  [1000000,     10000000]
DECORRELATION_TAUS2 = [10000000000, 100000000000, 1000000000]

DECORRELATION_TAU = 1000000000  # 100000000000  # 10000000000

ALL_PARAMS = [
    # (0, 1, 1),
    (1, 0, 1),
    (1, 1, 0),

    (1, 0, 0),
    # (0, 1, 0),
    # (0, 0, 1),
]

In [89]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer2

for params in ALL_PARAMS:
    key = params

    output_k = '-'.join(str(i) for i in key)
    res_file_path = SAVE_FOLDER + f'/ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue

    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1
            
            del result, phi
            model = None

        else:
            custom_regularizers = dict()
            
            if params[0]:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_model=prev_model._model,
                    topic_names=good_topic_names,
                )
                custom_regularizers[fix_regularizer.name] = fix_regularizer
            else:
                fix_regularizer = None
            
            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)
    
            bad_phi = deepcopy(bad_phi)

            if params[1]:
                decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_bad', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=bad_phi
                )
                custom_regularizers[decorr_bad_regularizer.name] = decorr_bad_regularizer
            else:
                decorr_bad_regularizer = None
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)

            if params[2]:
                decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_good', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=good_phi
                )
                custom_regularizers[decorr_good_regularizer.name] = decorr_good_regularizer
            else:
                decorr_good_regularizer = None
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # assert len(new_good_topic_names) > 0
            if len(new_good_topic_names) == 0:
                print('DOWNFALL: no good topics...')
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # assert set(good_topic_names) <= set(new_good_topic_names)
            if not (set(good_topic_names) <= set(new_good_topic_names)):
                print('DOWNFALL: some good topics lost...')
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

    for k, r in results.items():
        for s in r:
            s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

    for k, r in results.items():
        output_k = '-'.join(str(i) for i in k)
        with open(SAVE_FOLDER + f'/ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

(1, 0, 1)
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.4713878133278989
sparse_theta_sp: -3.380096644027831
decorrelation: 0.01
None
num_topics: {'good': 5, 'bad': 3, 'not_good': 15, 'total_bad': 3}
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9c4608fd90>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f9c48e123a0>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -0.6285170844371986
sparse_theta_sp: -4.506795525370441
decorrelation: 0.01
fix: 1000000000000
ext_decorr_good: 1000000000
SUCCESS: less bad topics
num_topics: {'good': 5, 'bad': 2, 'not_good': 15, 'total_bad': 5}
2


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9c46029670>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -1.0475284740619977
sparse_theta_sp: -7.511325875617401
decorrelation: 0.01
fix: 1000000000000
num_topics: {'good': 11, 'bad': 2, 'not_good': 9, 'total_bad': 42}
18


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9c6db30a60>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -1.0475284740619977
sparse_theta_sp: -7.511325875617401
decorrelation: 0.01
fix: 1000000000000
num_topics: {'good': 11, 'bad': 2, 'not_good': 9, 'total_bad': 44}
19


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f9c6d75c1f0>}
smooth_phi_bcg: 10.420151663037768
smooth_theta_bcg: 74.71792581535206
sparse_phi_sp: -1.0475284740619977
sparse_theta_sp: -7.511325875617401
decorrelation: 0.01
fix: 1000000000000
DOWNFALL: more bad topics...
num_topics: {'good': 11, 'bad': 3, 'not_good': 9, 'total_bad': 47}


In [91]:
1

1

In [92]:
! ls $SAVE_FOLDER

ablation_study			     iterative2_1000000000
decorrelation.json		     iterative2_10000000000
iterative_1000000		     iterative2_100000000000
iterative_10000000		     iterative2_100000000000.json
iterative_100000000		     iterative2_10000000000.json
iterative_100000000_unfinished.json  iterative2_1000000000.json
_iterative_10000000.json	     lda.json
iterative_10000000.json		     plsa.json
_iterative_1000000.json		     sparse.json
iterative_1000000.json		     tless.json


In [94]:
! ls $SAVE_FOLDER/ablation_study -alh

total 352K
drwxrwxr-x 2 alekseev_v mil_lab 4,0K мар 26 14:31 .
drwxrwxr-x 9 alekseev_v mil_lab 4,0K мар 25 22:48 ..
-rw-rw-r-- 1 alekseev_v mil_lab  26K мар 26 03:02 iterative_10000000_1-0-0.json
-rw-rw-r-- 1 alekseev_v mil_lab  14K мар 26 03:02 iterative_10000000_1-0-1.json
-rw-rw-r-- 1 alekseev_v mil_lab 8,8K мар 26 03:02 iterative_10000000_1-1-0.json
-rw-rw-r-- 1 alekseev_v mil_lab  26K мар 26 01:07 iterative_1000000_1-0-0.json
-rw-rw-r-- 1 alekseev_v mil_lab  26K мар 26 01:56 iterative_1000000_1-0-1.json
-rw-rw-r-- 1 alekseev_v mil_lab  25K мар 26 01:07 iterative_1000000_1-1-0.json
-rw-rw-r-- 1 alekseev_v mil_lab  26K мар 26 12:15 iterative2_100000000000_1-0-0.json
-rw-rw-r-- 1 alekseev_v mil_lab 7,5K мар 26 12:15 iterative2_100000000000_1-0-1.json
-rw-rw-r-- 1 alekseev_v mil_lab  10K мар 26 12:15 iterative2_100000000000_1-1-0.json
-rw-rw-r-- 1 alekseev_v mil_lab  26K мар 26 10:14 iterative2_10000000000_1-0-0.json
-rw-rw-r-- 1 alekseev_v mil_lab  25K мар 26 10:14 iterative2_1000000

In [96]:
! tail -n 20 $SAVE_FOLDER/ablation_study/iterative2_100000000000_1-0-1.json

            "9": 1.1756616093491055,
            "10": 0.9473087032704635,
            "11": 1.3298810340740868,
            "12": 0.9104903714039343,
            "13": 0.9400421582056122,
            "14": 1.099858425097038,
            "15": 0.9857446836345978,
            "16": 1.0894795043794931,
            "17": 0.9068202146464149,
            "18": 1.1678907238067384,
            "19": 1.2829860501661554
        },
        "num_topics": {
            "good": 18,
            "bad": 0,
            "not_good": 2,
            "total_bad": 5
        }
    }
]

In [ ]:
results.keys()

In [ ]:
for k, r in results.items():
    for s in r:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [ ]:
for k, r in results.items():
    output_k = '-'.join(str(i) for i in k)
    with open(SAVE_FOLDER + f'/ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )

In [ ]:
! ls results/20newsgroups/ablation_study

In [ ]:
for k, r in results.items():
    print(k)
    print(r[-1]['scores'])
    print(r[-1]['num_topics'])
    print()